In [1]:
import pandas as pd
import h3
import folium
from ssfaitk.models.effort.effort_classifier import EffortClassifier
from ssfaitk.utils.hexaGrid_pipeline import run_hex_aggregation, plot_hex_map

print("✅ All imports OK!")

✅ All imports OK!


In [2]:
df = pd.read_parquet(r"D:\Consultancy\WorldFish\Data\all_zanzibar_trips_jan26_predicted_parallel.parquet")

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst rows:")
df.head()

Shape: (20600395, 64)

Columns: ['Unnamed: 0', 'timestamp', 'Boat', 'trip_id', 'latitude', 'longitude', 'Speed (M/S)', 'Range (Meters)', 'Heading', 'Boat Name', 'Community', 'latitude_prev', 'longitude_prev', 'timestamp_prev', 'distance_km', 'dt_hours', 'speed_kmh', 'speed_prev', 'acceleration', 'accel_prev', 'jerk', 'longitude_next', 'latitude_next', 'bearing', 'bearing_prev', 'turn_angle', 'speed_kmh_mean', 'speed_kmh_std', 'speed_kmh_max', 'speed_kmh_min', 'acceleration_mean', 'acceleration_std', 'acceleration_max', 'acceleration_min', 'turn_angle_mean', 'turn_angle_std', 'turn_angle_max', 'turn_angle_min', 'speed_cv', 'accel_cv', 'dist_to_start_km', 'straightness', 'radius_gyration_km', 'sinuosity', 'hour', 'is_daytime', 'point_number', 'total_points', 'trip_position', 'is_fishing_speed', 'is_transit_speed', 'is_high_turning', 'is_low_straightness', 'is_high_sinuosity', 'is_clustered', 'is_variable_speed', 'is_fishing_pattern', 'is_search_pattern', 'fishing_score', 'is_fishing', 'd

,Unnamed: 0,timestamp,Boat,trip_id,latitude,longitude,Speed (M/S),Range (Meters),Heading,Boat Name,...,is_clustered,is_variable_speed,is_fishing_pattern,is_search_pattern,fishing_score,is_fishing,dist_to_shore_km,is_on_land,is_near_shore,is_fishing_filtered
939,10704,2024-07-29 02:54:00+00:00,24599.0,12067898,-6.17310,39.19338,0.00,0.000000,248.0,Muomba Mungu Hachoki,...,1,1,0,0,0.382353,0,999.0,False,0,0
940,10705,2024-07-29 02:54:02+00:00,24599.0,12067898,-6.17310,39.19335,1.66,3.320349,304.0,Muomba Mungu Hachoki,...,1,1,0,0,0.382353,0,999.0,False,0,0
941,10706,2024-07-29 02:55:06+00:00,24599.0,12067898,-6.17314,39.19331,0.09,8.921360,331.0,Muomba Mungu Hachoki,...,1,1,0,0,0.382353,0,999.0,False,0,0
942,10707,2024-07-29 02:56:21+00:00,24599.0,12067898,-6.17314,39.19334,0.04,6.258331,271.0,Muomba Mungu Hachoki,...,1,1,0,0,0.382353,0,999.0,False,0,0
943,10708,2024-07-29 02:57:28+00:00,24599.0,12067898,-6.17309,39.19334,0.08,4.563162,328.0,Muomba Mungu Hachoki,...,1,1,0,0,0.382353,0,999.0,False,0,0


In [3]:
from ssfaitk.utils.hexaGrid_pipeline import run_hex_aggregation, plot_hex_map

results, fishing_df = run_hex_aggregation(
    df,
    lat_col       = "latitude",
    lon_col       = "longitude",
    time_col      = "timestamp",
    trip_col      = "trip_id",
    speed_col     = "speed_kmh",
    fishing_col   = "is_fishing_filtered",  # use the filtered version
    resolutions   = [7, 8],
    output_dir    = r"D:\Consultancy\WorldFish\Data\hex_outputs"
)

H3 HEXAGONAL FISHING EFFORT AGGREGATION

[1/6] Detecting columns...
  latitude:  latitude
  longitude: longitude
  timestamp: timestamp
  trip_id:   trip_id
  speed:     speed_kmh
  fishing:   is_fishing_filtered

[2/6] Filtering data...
  Total points: 20,600,395
  Fishing points (is_fishing=1): 1,411,220

[3/6] Adding temporal dimensions...
  Year range: 2024-2026
  Seasons: {'NE_Monsoon': 864453, 'SE_Monsoon': 295687, 'Inter_Monsoon': 251080}
  Day/Night: {'night': 901043, 'day': 510177}

[4/6] Computing time-weighted intervals...
  Max interval cap: 600s (10 min)
  Total fishing effort: 18,326.2 hours (763.6 days)

[5/6] Assigning H3 hexagons...
  Assigning H3 resolution 7 (h3_res7)...
  Assigning H3 resolution 8 (h3_res8)...
  Resolution 7: 1,018 unique hexagons
  Resolution 8: 3,452 unique hexagons

[6/6] Aggregating metrics...

  Resolution 7 (h3_res7)

  [1/8] Overall aggregation...
        → 1,018 hexagons
  [2/8] By year...
        → 1,544 hexagon-year combinations
  [3/8] By

In [4]:
plot_hex_map(
    results["res7"]["overall"],
    metric      = "fishing_hours",
    title       = "Zanzibar Fishing Effort — Overall",
    output_path = r"D:\Consultancy\WorldFish\Data\hex_outputs\zanzibar_hex_map.html"
)

Plotting 974 hexagons colored by 'fishing_hours'...
✅ Saved: D:\Consultancy\WorldFish\Data\hex_outputs\zanzibar_hex_map.html (1.0 MB)


'D:\\Consultancy\\WorldFish\\Data\\hex_outputs\\zanzibar_hex_map.html'

In [5]:
import numpy as np

# Work on a copy so we don't touch the original
overall_log = results["res7"]["overall"].copy()
overall_log["fishing_hours_log"] = np.log1p(overall_log["fishing_hours"])  # log1p = log(1+x), handles zeros safely

plot_hex_map(
    overall_log,
    metric      = "fishing_hours_log",
    title       = "Zanzibar Fishing Effort — Log Scale",
    output_path = r"D:\Consultancy\WorldFish\Data\hex_outputs\zanzibar_hex_map_log.html",
    colormap    = "YlOrRd"
)

Plotting 974 hexagons colored by 'fishing_hours_log'...
✅ Saved: D:\Consultancy\WorldFish\Data\hex_outputs\zanzibar_hex_map_log.html (1.0 MB)


'D:\\Consultancy\\WorldFish\\Data\\hex_outputs\\zanzibar_hex_map_log.html'